# MLS Player Population and Transfermarkt Attribute Coverage

## Objective

Assess whether the Kaggle Transfermarkt dataset provides reliable player-level information for the MLS population represented in American Soccer Analysis (ASA) data, and identify additional data that must be acquired.

The audit focuses on four player characteristics:

- Date of birth, from which age can be derived at an observation date.
- Country of birth.
- Height.
- Preferred foot.

The analysis uses ASA salary and performance datasets as separate reference populations. It does not assume that the players represented in the salary data are identical to those represented in the performance datasets.

The audit distinguishes three cases:

1. Players matched to a unique Kaggle/Transfermarkt player record.
2. Players with multiple possible Kaggle/Transfermarkt matches.
3. Players with no exact name match in the Kaggle dataset.

A player absent from Kaggle is not necessarily absent from Transfermarkt itself.

The output is a coverage assessment and a list of player-level data gaps.

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
import numpy as np

RAW = Path("../../data/raw")
OUTPUT = Path("../../data/interim/transfermarkt_coverage")

OUTPUT.mkdir(parents=True, exist_ok=True)

KAGGLE_FILES = {
    "competitions": RAW / "competitions.csv",
    "clubs": RAW / "clubs.csv",
    "players": RAW / "players.csv",
}

ASA_FILES = {
    "salary": RAW / "american_soccer_analysis_mls_salaries_players.csv",
    "gplus_outfield": RAW / "american_soccer_analysis_mls_goals-added_players.csv",
    "gplus_goalkeepers": RAW / "american_soccer_analysis_mls_goals-added_goalkeepers.csv",
    "xg": RAW / "american_soccer_analysis_mls_xgoals_players.csv",
    "xpass": RAW / "american_soccer_analysis_mls_xpass_players.csv",
}

# The four characteristics being audited.
ATTRIBUTE_COLUMNS = {
    "date_of_birth": "date_of_birth",
    "country_of_birth": "country_of_birth",
    "height": "height_in_cm",
    "preferred_foot": "foot",
}

In [5]:
competitions = pd.read_csv(KAGGLE_FILES["competitions"])
clubs = pd.read_csv(KAGGLE_FILES["clubs"])

players = pd.read_csv(
    KAGGLE_FILES["players"],
    parse_dates=["date_of_birth", "contract_expiration_date"],
)

asa_tables = {
    source: pd.read_csv(path)
    for source, path in ASA_FILES.items()
    if path.exists()
}

print("Kaggle datasets")
print(f"Competitions: {competitions.shape}")
print(f"Clubs:        {clubs.shape}")
print(f"Players:      {players.shape}")

print("\nASA datasets")
for source, df in asa_tables.items():
    print(f"{source}: {df.shape}")

missing_files = [
    source
    for source, path in ASA_FILES.items()
    if not path.exists()
]

if missing_files:
    print("\nASA files not loaded. Check the filenames for:")
    print(missing_files)


for source, df in asa_tables.items():
    print(f"\n--- {source} ---")
    print(df.columns.tolist())
    print(df.head(3))

Kaggle datasets
Competitions: (65, 11)
Clubs:        (796, 17)
Players:      (50149, 26)

ASA datasets
salary: (15490, 8)
gplus_outfield: (6399, 13)
gplus_goalkeepers: (572, 12)
xg: (6970, 20)
xpass: (6970, 15)

--- salary ---
['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Base Salary', 'Guaranteed Compensation', 'Date']
   Unnamed: 0        Player Team  Season Position  Base Salary  \
0         NaN  Lionel Messi  MIA    2026        F  $25,000,000   
1         NaN  Lionel Messi  MIA    2023        M  $12,000,000   
2         NaN  Lionel Messi  MIA    2024        F  $12,000,000   

  Guaranteed Compensation        Date  
0             $28,333,332  2026-04-16  
1             $20,446,668  2023-09-15  
2             $20,446,668  2024-04-25  

--- gplus_outfield ---
['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Minutes', 'Dribbling', 'Fouling', 'Interrupting', 'Passing', 'Receiving', 'Shooting', 'Goals Added']
   Unnamed: 0           Player Team  Season Position  Minutes 

In [6]:
MLS_COMPETITION_ID = "MLS1"

mls_competitions = competitions[
    competitions["competition_id"].astype(str) == MLS_COMPETITION_ID
]

mls_clubs = clubs[
    clubs["domestic_competition_id"].astype(str) == MLS_COMPETITION_ID
].copy()

print("MLS competition metadata")
display(mls_competitions)

print(f"\nMLS club records: {len(mls_clubs)}")

display(
    mls_clubs[
        ["club_id", "name", "domestic_competition_id", "last_season"]
    ].sort_values("name")
)

print("Duplicate club IDs:", mls_clubs["club_id"].duplicated().sum())

display(
    mls_clubs.groupby("name", dropna=False)
    .agg(
        records=("club_id", "size"),
        club_ids=("club_id", "nunique"),
        last_seasons=("last_season", lambda x: sorted(x.dropna().astype(str).unique())),
    )
    .sort_values("records", ascending=False)
    .head(20)
)

MLS competition metadata


,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
38,MLS1,major-league-soccer,major-league-soccer,first_tier,domestic_league,184,United States,MLS1,amerika,30.0,https://www.transfermarkt.co.uk/major-league-s...



MLS club records: 30


,club_id,name,domestic_competition_id,last_season
556,51663,Atlanta United Football Club,MLS1,2025
691,72309,Austin FC,MLS1,2025
708,78435,Charlotte Football Club,MLS1,2025
507,432,Chicago Fire Soccer Club,MLS1,2025
665,69261,Club Internacional de Fútbol Miami,MLS1,2025
479,4078,Club de Foot Montréal,MLS1,2025
104,1247,Colorado Rapids,MLS1,2025
719,813,Columbus Crew,MLS1,2025
279,2440,D.C. United,MLS1,2025
751,8816,FC Dallas,MLS1,2025


Duplicate club IDs: 0


,records,club_ids,last_seasons
name,,,
Atlanta United Football Club,1,1,[2025]
Austin FC,1,1,[2025]
Toronto FC,1,1,[2025]
St. Louis City Soccer Club,1,1,[2025]
Sporting Kansas City,1,1,[2025]
Seattle Sounders FC,1,1,[2025]
San Jose Earthquakes,1,1,[2025]
San Diego Football Club,1,1,[2025]
Red Bull New York,1,1,[2025]


In [7]:
required_columns = ["Player", "Team", "Season"]

for source, df in asa_tables.items():
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(
            f"{source} is missing required columns: {missing}"
        )

def standardize_asa_population(source, df):
    columns = ["Player", "Team", "Season"]

    if "Position" in df.columns:
        columns.append("Position")

    population = df[columns].copy()

    population = population.rename(
        columns={
            "Player": "asa_player",
            "Team": "asa_team",
            "Season": "season",
            "Position": "asa_position",
        }
    )

    if "asa_position" not in population.columns:
        population["asa_position"] = pd.NA

    population["source"] = source

    population["season"] = (
        population["season"]
        .astype("string")
        .str.strip()
    )

    population["asa_player"] = (
        population["asa_player"]
        .astype("string")
        .str.strip()
    )

    population["asa_team"] = (
        population["asa_team"]
        .astype("string")
        .str.strip()
    )

    return population[
        [
            "source",
            "asa_player",
            "asa_team",
            "season",
            "asa_position",
        ]
    ]

asa_populations = pd.concat(
    [
        standardize_asa_population(source, df)
        for source, df in asa_tables.items()
    ],
    ignore_index=True,
)

asa_populations = asa_populations.drop_duplicates(
    subset=[
        "source",
        "asa_player",
        "asa_team",
        "season",
        "asa_position",
    ]
).reset_index(drop=True)

display(asa_populations.head())

print("ASA player-season records by source")
display(
    asa_populations.groupby("source").agg(
        records=("asa_player", "size"),
        players=("asa_player", "nunique"),
        seasons=("season", "nunique"),
        teams=("asa_team", "nunique"),
    )
)

source_season_counts = (
    asa_populations
    .groupby(["source", "season"])
    .agg(
        records=("asa_player", "size"),
        players=("asa_player", "nunique"),
    )
    .reset_index()
    .sort_values(["season", "source"])
)

display(source_season_counts)

source_player_seasons = {
    source: set(
        zip(
            df["asa_player"].dropna(),
            df.loc[df["asa_player"].notna(), "season"],
        )
    )
    for source, df in asa_populations.groupby("source")
}

source_overlap = []

for source_a in source_player_seasons:
    for source_b in source_player_seasons:
        if source_a >= source_b:
            continue

        a = source_player_seasons[source_a]
        b = source_player_seasons[source_b]

        source_overlap.append({
            "source_a": source_a,
            "source_b": source_b,
            "shared_player_seasons": len(a & b),
            "only_in_a": len(a - b),
            "only_in_b": len(b - a),
        })

source_overlap = pd.DataFrame(source_overlap)
display(source_overlap)

,source,asa_player,asa_team,season,asa_position
0,salary,Lionel Messi,MIA,2026,F
1,salary,Lionel Messi,MIA,2023,M
2,salary,Lionel Messi,MIA,2024,F
3,salary,Lionel Messi,MIA,2025,F
4,salary,Lorenzo Insigne,TOR,2023,M


ASA player-season records by source


,records,players,seasons,teams
source,,,,
gplus_goalkeepers,572,162,10,32
gplus_outfield,6399,2239,10,178
salary,9497,2829,11,30
xg,6970,2400,10,180
xpass,6970,2400,10,180


,source,season,records,players
0,gplus_goalkeepers,2016,43,43
10,gplus_outfield,2016,467,467
20,salary,2016,643,605
31,xg,2016,510,510
41,xpass,2016,510,510
1,gplus_goalkeepers,2017,49,49
11,gplus_outfield,2017,527,527
21,salary,2017,699,676
32,xg,2017,576,576
42,xpass,2017,576,576


,source_a,source_b,shared_player_seasons,only_in_a,only_in_b
0,gplus_goalkeepers,gplus_outfield,1,571,6396
1,gplus_goalkeepers,salary,568,4,8539
2,gplus_goalkeepers,xg,572,0,6396
3,gplus_goalkeepers,xpass,572,0,6396
4,gplus_outfield,salary,6280,117,2827
5,gplus_outfield,xg,6397,0,571
6,gplus_outfield,xpass,6397,0,571
7,salary,xg,6847,2260,121
8,salary,xpass,6847,2260,121
9,xg,xpass,6968,0,0


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA

asa_populations["match_name"] = (
    asa_populations["asa_player"]
)

asa_populations["normalized_name"] = (
    asa_populations["match_name"].map(normalize_name)
)

players["normalized_name"] = players["name"].map(normalize_name)

missing_tm_names = players["normalized_name"].isna()

players.loc[missing_tm_names, "normalized_name"] = (
    (
        players.loc[missing_tm_names, "first_name"].fillna("")
        + " "
        + players.loc[missing_tm_names, "last_name"].fillna("")
    )
    .map(normalize_name)
)

print("ASA records without a usable name:",
      asa_populations["normalized_name"].isna().sum())

print("Kaggle player records without a usable name:",
      players["normalized_name"].isna().sum())

ASA records without a usable name: 0
Kaggle player records without a usable name: 0


In [9]:
tm_candidates = (
    players.loc[players["normalized_name"].notna()]
    .groupby("normalized_name")
    .agg(
        candidate_count=("player_id", "nunique"),
        candidate_ids=("player_id", lambda x: list(pd.unique(x))),
        candidate_names=("name", lambda x: list(pd.unique(x))),
    )
    .reset_index()
)

asa_names = (
    asa_populations[
        ["asa_player", "match_name", "normalized_name"]
    ]
    .drop_duplicates()
    .copy()
)

name_matches = asa_names.merge(
    tm_candidates,
    on="normalized_name",
    how="left",
)

name_matches["candidate_count"] = (
    name_matches["candidate_count"].fillna(0).astype(int)
)

name_matches["match_status"] = np.select(
    [
        name_matches["candidate_count"].eq(0),
        name_matches["candidate_count"].eq(1),
        name_matches["candidate_count"].gt(1),
    ],
    [
        "unmatched",
        "unique_match",
        "ambiguous",
    ],
    default="unmatched",
)

display(
    name_matches["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .to_frame("unique_asa_names")
)

# Attach the unique Transfermarkt player ID only when there is exactly one candidate.
name_matches["player_id"] = name_matches.apply(
    lambda row: (
        row["candidate_ids"][0]
        if row["match_status"] == "unique_match"
        else pd.NA
    ),
    axis=1,
)

matched_population = asa_populations.merge(
    name_matches[
        [
            "asa_player",
            "normalized_name",
            "candidate_count",
            "candidate_ids",
            "match_status",
            "player_id",
        ]
    ],
    on=["asa_player", "normalized_name"],
    how="left",
)

matched_population["player_id"] = matched_population["player_id"].astype("Int64")

display(matched_population.head())

,unique_asa_names
match_status,
unique_match,1632
unmatched,1155
ambiguous,93


,source,asa_player,asa_team,season,asa_position,match_name,normalized_name,candidate_count,candidate_ids,match_status,player_id
0,salary,Lionel Messi,MIA,2026,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003
1,salary,Lionel Messi,MIA,2023,M,Lionel Messi,lionel messi,1,[28003],unique_match,28003
2,salary,Lionel Messi,MIA,2024,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003
3,salary,Lionel Messi,MIA,2025,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003
4,salary,Lorenzo Insigne,TOR,2023,M,Lorenzo Insigne,lorenzo insigne,1,[133964],unique_match,133964


In [10]:
matching_summary = (
    matched_population
    .groupby(["source", "match_status"])
    .agg(
        player_season_records=("asa_player", "size"),
        unique_players=("asa_player", "nunique"),
    )
    .reset_index()
)

display(
    matching_summary.sort_values(["source", "match_status"])
)

matching_rates = (
    matched_population
    .groupby("source")
    .agg(
        player_season_records=("asa_player", "size"),
        unique_players=("asa_player", "nunique"),
        unique_matches=("player_id", "nunique"),
        unmatched_records=("match_status", lambda x: x.eq("unmatched").sum()),
        ambiguous_records=("match_status", lambda x: x.eq("ambiguous").sum()),
        matched_records=("match_status", lambda x: x.eq("unique_match").sum()),
    )
    .reset_index()
)

matching_rates["unique_match_rate"] = (
    matching_rates["matched_records"]
    / matching_rates["player_season_records"]
)

display(matching_rates)

ambiguous_players = name_matches[
    name_matches["match_status"] == "ambiguous"
].copy()

display(
    ambiguous_players[
        [
            "asa_player",
            "normalized_name",
            "candidate_count",
            "candidate_ids",
            "candidate_names",
        ]
    ].sort_values("asa_player")
)

unmatched_players = name_matches[
    name_matches["match_status"] == "unmatched"
].copy()

display(
    unmatched_players[
        ["asa_player", "normalized_name"]
    ].sort_values("asa_player")
)

,source,match_status,player_season_records,unique_players
0,gplus_goalkeepers,ambiguous,4,2
1,gplus_goalkeepers,unique_match,385,98
2,gplus_goalkeepers,unmatched,183,62
3,gplus_outfield,ambiguous,236,83
4,gplus_outfield,unique_match,4064,1308
5,gplus_outfield,unmatched,2099,848
6,salary,ambiguous,301,92
7,salary,unique_match,6015,1618
8,salary,unmatched,3181,1119
9,xg,ambiguous,240,85


,source,player_season_records,unique_players,unique_matches,unmatched_records,ambiguous_records,matched_records,unique_match_rate
0,gplus_goalkeepers,572,162,98,183,4,385,0.673077
1,gplus_outfield,6399,2239,1308,2099,236,4064,0.635099
2,salary,9497,2829,1617,3181,301,6015,0.633358
3,xg,6970,2400,1405,2282,240,4448,0.638164
4,xpass,6970,2400,1405,2282,240,4448,0.638164


,asa_player,normalized_name,candidate_count,candidate_ids,candidate_names
685,Alan Franco,alan franco,2,"[400960, 503343]",[Alan Franco]
1648,Alex,alex,7,"[13213, 15420, 18537, 86811, 140957, 418752, 6...",[Alex]
2217,Alex Harris,alex harris,2,"[192500, 1356045]",[Alex Harris]
372,André Gomes,andre gomes,2,"[221025, 670710]",[André Gomes]
688,Antony,antony,2,"[602105, 798687]",[Antony]
...,...,...,...,...,...
386,Sergio Santos,sergio santos,3,"[370561, 537759, 557185]","[Sergio Santos, Sérgio Santos]"
797,Tiago,tiago,2,"[993685, 1083095]",[Tiago]
735,Tommy Smith,tommy smith,3,"[12401, 54381, 121398]",[Tommy Smith]
1400,Zeca,zeca,2,"[131161, 325196]",[Zeca]


,asa_player,normalized_name
2441,A.J. Cochran,a j cochran
1329,A.J. DeLaGarza,a j delagarza
2796,Aaron Jones,aaron jones
2575,Aaron Kovar,aaron kovar
1587,Aaron Maund,aaron maund
...,...,...
2692,Álvaro Quezada,alvaro quezada
891,Álvaro Saborio,alvaro saborio
2495,Éver Alvarado,ever alvarado
2800,Øyvind Alseth,yvind alseth


In [11]:
tm_attributes = players[
    [
        "player_id",
        "name",
        "date_of_birth",
        "country_of_birth",
        "height_in_cm",
        "foot",
    ]
].copy()

unique_matches = matched_population[
    matched_population["match_status"] == "unique_match"
].copy()

unique_matches = unique_matches.merge(
    tm_attributes,
    on="player_id",
    how="left",
    validate="many_to_one",
)

display(unique_matches.head())

attribute_fields = {
    "date_of_birth": "date_of_birth",
    "country_of_birth": "country_of_birth",
    "height": "height_in_cm",
    "preferred_foot": "foot",
}

attribute_summary = []

for source, df in unique_matches.groupby("source"):
    for attribute, column in attribute_fields.items():
        total = len(df)
        available = df[column].notna().sum()

        attribute_summary.append({
            "source": source,
            "attribute": attribute,
            "matched_player_season_records": total,
            "records_with_attribute": available,
            "records_missing_attribute": total - available,
            "coverage_rate": available / total if total else np.nan,
        })

attribute_summary = pd.DataFrame(attribute_summary)

display(
    attribute_summary.sort_values(["source", "attribute"])
)

unique_player_attributes = (
    unique_matches
    .drop_duplicates(subset=["source", "player_id"])
)

unique_player_attribute_summary = []

for source, df in unique_player_attributes.groupby("source"):
    for attribute, column in attribute_fields.items():
        total = len(df)
        available = df[column].notna().sum()

        unique_player_attribute_summary.append({
            "source": source,
            "attribute": attribute,
            "unique_matched_players": total,
            "players_with_attribute": available,
            "players_missing_attribute": total - available,
            "coverage_rate": available / total if total else np.nan,
        })

unique_player_attribute_summary = pd.DataFrame(
    unique_player_attribute_summary
)

display(
    unique_player_attribute_summary.sort_values(
        ["source", "attribute"]
    )
)

,source,asa_player,asa_team,season,asa_position,match_name,normalized_name,candidate_count,candidate_ids,match_status,player_id,name,date_of_birth,country_of_birth,height_in_cm,foot
0,salary,Lionel Messi,MIA,2026,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003,Lionel Messi,1987-06-24,Argentina,170.0,left
1,salary,Lionel Messi,MIA,2023,M,Lionel Messi,lionel messi,1,[28003],unique_match,28003,Lionel Messi,1987-06-24,Argentina,170.0,left
2,salary,Lionel Messi,MIA,2024,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003,Lionel Messi,1987-06-24,Argentina,170.0,left
3,salary,Lionel Messi,MIA,2025,F,Lionel Messi,lionel messi,1,[28003],unique_match,28003,Lionel Messi,1987-06-24,Argentina,170.0,left
4,salary,Lorenzo Insigne,TOR,2023,M,Lorenzo Insigne,lorenzo insigne,1,[133964],unique_match,133964,Lorenzo Insigne,1991-06-04,Italy,163.0,right


,source,attribute,matched_player_season_records,records_with_attribute,records_missing_attribute,coverage_rate
1,gplus_goalkeepers,country_of_birth,385,385,0,1.000000
0,gplus_goalkeepers,date_of_birth,385,385,0,1.000000
2,gplus_goalkeepers,height,385,385,0,1.000000
3,gplus_goalkeepers,preferred_foot,385,245,140,0.636364
5,gplus_outfield,country_of_birth,4064,4034,30,0.992618
4,gplus_outfield,date_of_birth,4064,4064,0,1.000000
6,gplus_outfield,height,4064,4034,30,0.992618
7,gplus_outfield,preferred_foot,4064,3547,517,0.872785
9,salary,country_of_birth,6015,5955,60,0.990025
8,salary,date_of_birth,6015,6015,0,1.000000


,source,attribute,unique_matched_players,players_with_attribute,players_missing_attribute,coverage_rate
1,gplus_goalkeepers,country_of_birth,98,98,0,1.000000
0,gplus_goalkeepers,date_of_birth,98,98,0,1.000000
2,gplus_goalkeepers,height,98,98,0,1.000000
3,gplus_goalkeepers,preferred_foot,98,66,32,0.673469
5,gplus_outfield,country_of_birth,1308,1291,17,0.987003
4,gplus_outfield,date_of_birth,1308,1308,0,1.000000
6,gplus_outfield,height,1308,1287,21,0.983945
7,gplus_outfield,preferred_foot,1308,1153,155,0.881498
9,salary,country_of_birth,1617,1582,35,0.978355
8,salary,date_of_birth,1617,1617,0,1.000000


In [12]:
unresolved_players = (
    matched_population[
        matched_population["match_status"].isin(
            ["unmatched", "ambiguous"]
        )
    ]
    .groupby(["source", "asa_player", "match_status"])
    .agg(
        seasons=("season", lambda x: sorted(x.dropna().unique().tolist())),
        teams=("asa_team", lambda x: sorted(x.dropna().unique().tolist())),
    )
    .reset_index()
)

display(unresolved_players)

field_gaps = []

for _, row in unique_player_attributes.iterrows():
    missing_attributes = [
        attribute
        for attribute, column in attribute_fields.items()
        if pd.isna(row[column])
    ]

    for attribute in missing_attributes:
        field_gaps.append({
            "source": row["source"],
            "asa_player": row["asa_player"],
            "player_id": row["player_id"],
            "attribute": attribute,
        })

field_gaps = pd.DataFrame(
    field_gaps,
    columns=["source", "asa_player", "player_id", "attribute"],
)

display(field_gaps)

player_data_gaps = (
    unique_player_attributes[
        ["source", "asa_player", "player_id"]
        + list(attribute_fields.values())
    ]
    .copy()
)

player_data_gaps["missing_attributes"] = player_data_gaps.apply(
    lambda row: [
        attribute
        for attribute, column in attribute_fields.items()
        if pd.isna(row[column])
    ],
    axis=1,
)

player_data_gaps = player_data_gaps[
    player_data_gaps["missing_attributes"].map(bool)
]

display(player_data_gaps)

,source,asa_player,match_status,seasons,teams
0,gplus_goalkeepers,Abraham Rodriguez,unmatched,"[2022, 2023]",[COL]
1,gplus_goalkeepers,Abraham Romero,unmatched,[2024],[CLB]
2,gplus_goalkeepers,Adam Grinwis,unmatched,"[2018, 2021]",[ORL]
3,gplus_goalkeepers,Adam Larsen Kwarasey,unmatched,[2016],[POR]
4,gplus_goalkeepers,Alex Tambakis,unmatched,[2017],[ATL]
...,...,...,...,...,...
4191,xpass,Álvaro Quezada,unmatched,[2021],[LAFC]
4192,xpass,Álvaro Saborio,unmatched,[2016],[DCU]
4193,xpass,Éver Alvarado,unmatched,[2016],[SKC]
4194,xpass,Øyvind Alseth,unmatched,[2017],[TOR]


,source,asa_player,player_id,attribute
0,salary,Bastian Schweinsteiger,2514,height
1,salary,Bastian Schweinsteiger,2514,preferred_foot
2,salary,Djordje Mihailovic,484756,preferred_foot
3,salary,Henry Kessler,638732,preferred_foot
4,salary,Justen Glad,334597,preferred_foot
...,...,...,...,...
978,xpass,Sharod George,1062210,preferred_foot
979,xpass,Yair Arboleda,352708,preferred_foot
980,xpass,Eric Klein,1150822,country_of_birth
981,xpass,Eric Klein,1150822,height


,source,asa_player,player_id,date_of_birth,country_of_birth,height_in_cm,foot,missing_attributes
41,salary,Bastian Schweinsteiger,2514,1984-08-01,Germany,NaN,NaN,"[height, preferred_foot]"
195,salary,Djordje Mihailovic,484756,1998-11-10,United States,178.0,NaN,[preferred_foot]
563,salary,Henry Kessler,638732,1998-06-25,United States,193.0,NaN,[preferred_foot]
566,salary,Justen Glad,334597,1997-02-28,United States,183.0,NaN,[preferred_foot]
662,salary,Tim Parker,354737,1993-02-23,United States,188.0,NaN,[preferred_foot]
...,...,...,...,...,...,...,...,...
19299,xpass,Baye Coulibaly,1132631,2006-01-08,Mali,185.0,NaN,[preferred_foot]
19301,xpass,Aiden Hezarkhani,1242126,2007-06-28,United States,NaN,NaN,"[height, preferred_foot]"
19333,xpass,Sharod George,1062210,2004-05-10,NaN,NaN,NaN,"[country_of_birth, height, preferred_foot]"
19336,xpass,Yair Arboleda,352708,1996-04-07,Colombia,178.0,NaN,[preferred_foot]


In [13]:
all_asa_players = (
    asa_populations[
        ["source", "asa_player", "season"]
    ]
    .drop_duplicates()
)

player_source_presence = (
    all_asa_players
    .assign(present=1)
    .pivot_table(
        index=["asa_player", "season"],
        columns="source",
        values="present",
        fill_value=0,
        aggfunc="max",
    )
    .reset_index()
)

display(player_source_presence.head(20))

source_presence_summary = (
    player_source_presence
    .drop(columns=["asa_player", "season"])
    .sum()
    .rename("player_season_records")
    .to_frame()
)

display(source_presence_summary)

source,asa_player,season,gplus_goalkeepers,gplus_outfield,salary,xg,xpass
0,A.J. Cochran,2020,0,0,1,0,0
1,A.J. DeLaGarza,2016,0,1,1,1,1
2,A.J. DeLaGarza,2017,0,1,1,1,1
3,A.J. DeLaGarza,2018,0,1,1,1,1
4,A.J. DeLaGarza,2019,0,1,1,1,1
5,A.J. DeLaGarza,2020,0,1,1,1,1
6,A.J. DeLaGarza,2021,0,1,1,1,1
7,A.J. DeLaGarza,2022,0,1,1,1,1
8,AJ Marcucci,2021,0,0,1,0,0
9,AJ Marcucci,2022,0,0,1,0,0


,player_season_records
source,
gplus_goalkeepers,572
gplus_outfield,6397
salary,9107
xg,6968
xpass,6968


In [14]:
matching_rates.to_csv(
    OUTPUT / "matching_summary_by_source.csv",
    index=False,
)

matching_summary.to_csv(
    OUTPUT / "matching_status_by_source.csv",
    index=False,
)

name_matches.to_csv(
    OUTPUT / "asa_to_transfermarkt_name_matches.csv",
    index=False,
)

unmatched_players.to_csv(
    OUTPUT / "unmatched_asa_players.csv",
    index=False,
)

ambiguous_players.to_csv(
    OUTPUT / "ambiguous_asa_players.csv",
    index=False,
)

attribute_summary.to_csv(
    OUTPUT / "attribute_coverage_player_seasons.csv",
    index=False,
)

unique_player_attribute_summary.to_csv(
    OUTPUT / "attribute_coverage_unique_players.csv",
    index=False,
)

player_data_gaps.to_csv(
    OUTPUT / "matched_player_attribute_gaps.csv",
    index=False,
)

unresolved_players.to_csv(
    OUTPUT / "unresolved_player_acquisition_list.csv",
    index=False,
)

player_source_presence.to_csv(
    OUTPUT / "asa_player_source_presence.csv",
    index=False,
)

print(f"Audit outputs saved to: {OUTPUT.resolve()}")

Audit outputs saved to: /Users/NATGOMEZ/repos/player-value-model/data/interim/transfermarkt_coverage


## Findings

This notebook explored whether the [Kaggle Transfermarkt dataset](https://www.kaggle.com/datasets/davidcariboo/player-scores?select=transfers.csv) could provide player characteristics for the MLS salary and performance analysis.

The audit revealed two important findings:

1. **Player matching:** Matching players by name introduces ambiguity. The Kaggle transfers table could help resolve these cases by identifying which players have a history of playing in MLS.

2. **Population discrepancies:** Some players appear in ASA salary records but not in performance datasets. Manual inspection suggests that many either did not play an MLS match or participated in reserve-team competitions such as the USL Championship. These competitions have different competitive contexts, making their performance data unsuitable for direct comparison.

Further investigation showed that the ASA Python package already provides player names, birth dates, heights, and nationalities, eliminating the need to integrate a separate source for these characteristics.

## Decisions

**ASA will be the sole data source**, simplifying player identification, integration, and cleaning.

The analytical population will include only players with recorded MLS minutes, ensuring that performance observations come from the same competition.

The analysis will remain centered on player performance. Age will be retained as a potential explanatory variable, while height may be relevant for specific positions, particularly goalkeepers and defenders. Preferred foot will be excluded from the analysis.

The Kaggle Transfermarkt dataset will remain available as a reference but will not be incorporated into the final analytical pipeline.